# Entrenar YOLOv8 en Colab: detección de formas geométricas

1. Ve a **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. Ejecuta las celdas en orden.
3. Al final se descarga `resultados.zip`, con `models/best.pt`, `models/metricas.json` y las gráficas. Copia la carpeta `models/` dentro del proyecto en tu PC y ejecuta allá `demo_webcam.py`.

## 1. Subir el proyecto
Sube el archivo `proyecto_yolo_formas.zip`.

In [ ]:
from google.colab import files
subido = files.upload()          # elige proyecto_yolo_formas.zip
!unzip -q -o proyecto_yolo_formas.zip
%cd proyecto_yolo_formas

## 2. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt
import torch; print('GPU disponible:', torch.cuda.is_available())

## 3. Generar el dataset (1200 imágenes, 70/20/10, formato YOLO)
Primero se descargan 128 fotos reales (COCO128). Se usan como fondo y como imágenes **negativas** sin figuras, para que el modelo no confunda caras, manos u objetos con figuras.

In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip && unzip -q -o coco128.zip
!python data/generar_dataset.py --n 1200 --fondos coco128/images/train2017

Vista de algunas imágenes con sus etiquetas, para la parte del **dataset** en la presentación:

In [ ]:
import cv2, glob, matplotlib.pyplot as plt
CLASES = ['circulo', 'cuadrado', 'triangulo']
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, f in zip(ax.flat, sorted(glob.glob('data/formas/images/train/*.jpg'))[:8]):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); h, w = img.shape[:2]
    for l in open(f.replace('images', 'labels').replace('.jpg', '.txt')).read().split('\n'):
        if not l.strip(): continue  # imagen negativa (sin figuras)
        c, xc, yc, bw, bh = map(float, l.split())
        x1, y1, x2, y2 = int((xc-bw/2)*w), int((yc-bh/2)*h), int((xc+bw/2)*w), int((yc+bh/2)*h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 3)
        cv2.putText(img, CLASES[int(c)], (x1, max(15, y1-6)), 0, 0.9, (0, 120, 0), 2)
    a.imshow(img); a.axis('off')
plt.tight_layout(); plt.savefig('muestras_dataset.png'); plt.show()

## 4. Entrenar (imgsz=640, hasta 60 épocas, early stopping con patience=15)
Con GPU T4 tarda unos 20–30 minutos. Revisa que en la columna **GPU_mem** no diga `0G`.

In [ ]:
!python train.py --epochs 60 --imgsz 640 --batch 16

In [ ]:
from IPython.display import Image, display
display(Image('runs/detect/formas/results.png', width=900))   # pérdidas y mAP por época

## 5. Validar: mAP@0.5, precisión, recall, matriz de confusión y Accuracy_val

In [ ]:
!python validate.py

In [ ]:
display(Image('runs/detect/validacion/confusion_matrix.png', width=600))

## 6. Descargar resultados

In [ ]:
!zip -q -r resultados.zip models runs muestras_dataset.png
files.download('resultados.zip')